# this notebook contains pre-processing for the clustering task with HDBSCAN
Lemmatizer and BoW were the best combination when using KMEANS (tested in KMEANS pre processing)
Because of how memory expensive HDBSCAN is, in addition to the steps taken in KMEANS pre-processing, sampling a small subset of the dataset is needed. For the same reason we'll delete more stopwords and delete more "rare words"

In [ ]:
%pip install nltk
import nltk
nltk.download('punkt')
nltk.download('wordnet')
%pip install langdetect

In [ ]:
import pandas as pd
import numpy as np

#to try executing HDBSCAN on the subsample to see if it works without memory issues (and how much time it takes)
from sklearn.cluster import HDBSCAN

#to time the execution of HDBSCAN
import time

In [ ]:
# #This is the same steps taken in KMEANS-pre-processing to avoid executing again the language detection
# def remove_rare_words(data):
#     all_words = ' '.join(data['body']).split()
#     word_freq = pd.Series(all_words).value_counts()
#     rare_words = word_freq[word_freq == 1].index
#     data['body'] = data['body'].apply(lambda x: ' '.join([word for word in x.split() if word not in rare_words]))
#     del all_words, word_freq, rare_words
#     return data

# def remove_stopwords(numerStopwords, df):
#     MostFrequentWords = df['body'].str.split().explode().value_counts()[0:numerStopwords].index.tolist()
#     #print(MostFrequentWords)
#     #removing most frequent words
#     for word in MostFrequentWords:
#         df['body']=df['body'].str.replace(rf'\b{word}\b', '', regex=True)
#     return df

# data = pd.read_csv('data_unsupervised.csv')

# data['body'] = data['body'].str.lower()
# data['body'] = data['body'].str.replace(r'http\S+|www\.\S+', '', regex=True)
# data['body'] = data['body'].str.replace(r'[^a-z\s]', '', regex=True)
# data = data[data['body'].str.strip().astype(bool)]
# data = data.dropna()
# data=data.drop_duplicates()

# data = remove_rare_words(data)

# data = data[data['body'].str.strip().astype(bool)]

# langs = []
# with open("unsupervised_lang.txt") as f:
#     for line in f:
#         langs.append(line.strip())
#     data = data[pd.Series(langs, index=data.index)=='en']

# data = remove_stopwords(150, data)
# data = data.drop_duplicates("body")

data = pd.read_csv(f"without_stopwords_150.csv")
# Fixed
# #we actually forgot to remove duplicates/empty rows after removing stopwords, so "we do it now" by creating a mask
# mask=[]
# for index, row in data.iterrows():
#     try:
#         if len(row['body'].strip()) == 0:
#             mask.append(False)
#         else:            
#             mask.append(True)
#     except:#if NaN
#         mask.append(False)
# data = data[mask]

In [ ]:
#how many comments have less than 10 words?
num_short_comments = len(data[data['body'].apply( lambda x : len(x.split())) < 10])
print(f"Number of comments with less than 10 words: {num_short_comments} ({num_short_comments/len(data)*100:.2f}%)")
num_short_comments = len(data[data['body'].apply( lambda x : len(x.split())) < 5])
print(f"Number of comments with less than 5 words: {num_short_comments} ({num_short_comments/len(data)*100:.2f}%)")
num_short_comments = len(data[data['body'].apply( lambda x : len(x.split())) < 3])
print(f"Number of comments with less than 3 words: {num_short_comments} ({num_short_comments/len(data)*100:.2f}%)")
num_short_comments = len(data[data['body'].apply( lambda x : len(x.split())) == 1])
print(f"Number of comments with only 1 word: {num_short_comments} ({num_short_comments/len(data)*100:.2f}%)")

Number of comments with less than 10 words: 406070 (54.92%)
Number of comments with less than 5 words: 193320 (26.15%)
Number of comments with less than 3 words: 70502 (9.54%)
Number of comments with only 1 word: 18779 (2.54%)


Ideally we would like to keep subreddit and authors proportion in the subsample, while also avoiding very short comments

In [ ]:
def get_subsample_mask(data, n_samples):
    data_copy = data.copy()
    data_copy['word_count'] = data_copy['body'].str.split().apply(len)

    #Stratified sampling by subreddit, preferring longer comments
    #We'll sort by word_count descending, then group and sample
    data_copy = data_copy.sort_values('word_count', ascending=False)

    #n_samples = 20000
    group_sizes = data_copy.groupby('subreddit').size()
    group_frac = n_samples / group_sizes.sum()

    #Sample proportionally from each subreddit, but ensure at least 1 sample per subreddit
    sampled_indices = (
        data_copy
        .groupby('subreddit', group_keys=False)
        .apply(lambda x: x.head(max(1, int(len(x) * group_frac))))
        .index
    )
    
    #boolean mask for the original data
    mask = data.index.isin(sampled_indices)
    return mask

In [ ]:
subsample = get_subsample_mask(data, 20000)

#X = pd.DataFrame(np.load("X_reduced_150_stemmer_BoW.npy"))[mask]
X = pd.DataFrame(np.load("X_reduced_150_stemmer_BoW.npy"))

X = X[subsample]

start_time = time.time()
clusterer = HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean', cluster_selection_method='eom')
clusterer.fit(X)
end_time = time.time()
#print the time taken
print(f"HDBSCAN took {round((end_time - start_time)/60)} minutes on the subsample of 20,000 comments")

/tmp/ipykernel_5523/1223357000.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.head(max(1, int(len(x) * group_frac))))


In [ ]:
# #If the time takes is reasonable
# subsample_dimensions = [] 
# for i in subsample_dimensions:
#     subsample = get_subsample_mask(data, i)

#     X = pd.read_csv("without_stopwords_150.csv")
#     X = X[subsample].reset_index(drop=True)

#     start_time = time.time()
#     clusterer = HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean', cluster_selection_method='eom')
#     clusterer.fit(X)
#     end_time = time.time()
#     print(f"HDBSCAN took {end_time - start_time:.2f} seconds on the subsample of {i} comments")

In [ ]:
#save the subsample mask 
np.save("subsample_mask.npy", subsample)